In [208]:
from collections import defaultdict
from tabulate import tabulate

In [209]:
N = 8
# Initial board state
board = [[None, None, None, None, None, 37, None, 1100],
         [None, None, None, None, None, None, None, None],
         [None, None, None, 23, None, 138, None, None],
         [528, None, None, None, None, None, None, None],
         [None, 449, None, None, 16, None, None, None],
         [None, 750, None, 88, None, 272, 1, None],
         [None, None, None, None, None, None, None, None],
         [0, None, None, None, None, None, None, None]]

regions = [[1, 1, 1, 1, 1, 2, 2, 2],
           [3, 3, 3, 4, 4, 5, 5, 2],
           [3, 6, 3, 4, 4, 4, 5, 2],
           [7, 6, 6, 8, 8, 9, 5, 5],
           [7, 7, 6, 8, 8, 9, 9, 10],
           [7, 11, 6, 12, 9, 9, 13, 10],
           [7, 11, 12, 12, 12, 13, 13, 10],
           [11, 11, 11, 12, 13, 13, 10, 10]]

print(tabulate(regions))
print(tabulate(board))


--  --  --  --  --  --  --  --
 1   1   1   1   1   2   2   2
 3   3   3   4   4   5   5   2
 3   6   3   4   4   4   5   2
 7   6   6   8   8   9   5   5
 7   7   6   8   8   9   9  10
 7  11   6  12   9   9  13  10
 7  11  12  12  12  13  13  10
11  11  11  12  13  13  10  10
--  --  --  --  --  --  --  --
---  ---    --  --  ---  -  ----
                     37     1100

            23      138
528
     449        16
     750    88      272  1

  0
---  ---    --  --  ---  -  ----


In [210]:
level_knight_moves = [
    (2, 1), (2, -1), (-2, 1), (-2, -1),
    (1, 2), (1, -2), (-1, 2), (-1, -2)
]

tower_knight_moves = [(2, 0), (-2, 0), (0, -2), (0, 2)]

def in_bounds(r, c):
    return 0 <= r < N and 0 <= c < N

def is_checkpoint(n, k = 7):
    """
    Checks if we are at a checkpoint where the knight wrote down a move
    Assumes k=7 through trial and error
    """
    return (n <= 18 and n % 3 == 0) or (n > 18 and (n - 18) % k == 0)

In [211]:
def legal_moves(position, visited, region_has_tower, curr_board, num_moves, curr_score, level):
    """
    Generates all legal moves given the board state and assumed k = 7

    :return: Returns a legal move: (new position, type of move, new score and level)
    """
    r, c = position
    next_move = num_moves + 1

    for moves, kind in ((level_knight_moves, "L"), (tower_knight_moves, "T")):
        for dr, dc in moves:
            nr, nc = r + dr, c + dc

            if not in_bounds(nr, nc) or (nr, nc) in visited:
                continue

            bval = board[nr][nc]

            # Level move (maintains the same level)
            if kind == "L":
                if level == 1 and region_has_tower[regions[nr][nc]]:
                    continue

                new_score = curr_score + next_move
                new_level = level
                move_type = "L"

            # Tower move (moves up or down)
            else:
                if level == 1:
                    move_type = "D"
                    new_level = level - 1
                    new_score = curr_score / next_move
                elif not region_has_tower[regions[nr][nc]]:
                    move_type = "U"
                    new_level = level + 1
                    new_score = curr_score * next_move
                else:
                    continue

                if not new_score.is_integer():
                    continue

            # Checkpoint logic
            if is_checkpoint(next_move):
                if new_score == bval:
                    yield (nr, nc, move_type, new_score, new_level)
                continue

            if not bval:
                yield (nr, nc, move_type, new_score, new_level)


In [212]:
# Testing application of legal moves function
position = (7,0)
visited = {(7,0)}
curr_board = board
num_moves = 0
level = 1
curr_score = 0
region_has_tower = defaultdict(bool)

moves = legal_moves(position, visited, region_has_tower, curr_board, num_moves, curr_score, level)

for move in moves:
    print(move)

(6, 2, 'L', 1, 1)
(5, 0, 'D', 0.0, 0)
(7, 2, 'D', 0.0, 0)


In [213]:
def search(board, visited, history, region_has_tower, num_moves, curr_score, level):
    """
    Searches for solution by backtracking

    :return: Generator for solution set
    """

    moves_found = False
    curr_position = history[-1][:2]

    moves = legal_moves(curr_position, visited, region_has_tower, board, num_moves, curr_score, level)

    for (nr, nc, move_type, new_score, new_level) in moves:
        moves_found = True

        # Apply move
        history.append((nr, nc, num_moves + 1, move_type, new_score, new_level))
        visited.add((nr, nc))
        prev_val = board[nr][nc]
        board[nr][nc] = new_score

        # Add tower to board
        region_idx = regions[nr][nc]
        prev_region_state = region_has_tower[region_idx]

        if move_type == "L" and new_level == 1:
            region_has_tower[region_idx] = True

        elif move_type == "U":
            region_has_tower[region_idx] = True

        # Recurse
        yield from search(
            board,
            visited,
            history,
            region_has_tower,
            num_moves + 1,
            new_score,
            new_level,
        )

        # Backtrack
        history.pop()
        visited.remove((nr, nc))
        board[nr][nc] = prev_val
        region_has_tower[region_idx] = prev_region_state

    # Terminal
    if not moves_found:
        # Check if valid solution
        if sum(region_has_tower.values()) == 13 and (num_moves) >= 53:
            yield list(history), [row[:] for row in board]
        return None

    return None



In [214]:
board = [[None, None, None, None, None, 37, None, 1100],
         [None, None, None, None, None, None, None, None],
         [None, None, None, 23, None, 138, None, None],
         [528, None, None, None, None, None, None, None],
         [None, 449, None, None, 16, None, None, None],
         [None, 750, None, 88, None, 272, 1, None],
         [None, None, None, None, None, None, None, None],
         [0, None, None, None, None, None, None, None]]

starting_position = (7,0, 0, "S", 0, 1)
region_has_tower = defaultdict(bool)
r, c = starting_position[:2]
region_has_tower[regions[r][c]] = True

sols = search(board, {starting_position}, [starting_position], region_has_tower, 0, 0, 1)

for history, final_board in sols:
    for move in history:
        print(move, "\n")
    print(tabulate(board))
    final_board_ = final_board


(7, 0, 0, 'S', 0, 1) 

(6, 2, 1, 'L', 1, 1) 

(5, 4, 2, 'L', 3, 1) 

(5, 6, 3, 'D', 1.0, 0) 

(7, 7, 4, 'L', 5.0, 0) 

(6, 5, 5, 'L', 10.0, 0) 

(4, 4, 6, 'L', 16.0, 0) 

(2, 4, 7, 'U', 112.0, 1) 

(0, 4, 8, 'D', 14.0, 0) 

(2, 3, 9, 'L', 23.0, 0) 

(0, 2, 10, 'L', 33.0, 0) 

(1, 0, 11, 'L', 44.0, 0) 

(3, 0, 12, 'U', 528.0, 1) 

(1, 1, 13, 'L', 541.0, 1) 

(0, 3, 14, 'L', 555.0, 1) 

(0, 5, 15, 'D', 37.0, 0) 

(1, 3, 16, 'L', 53.0, 0) 

(3, 2, 17, 'L', 70.0, 0) 

(5, 3, 18, 'L', 88.0, 0) 

(6, 1, 19, 'L', 107.0, 0) 

(4, 0, 20, 'L', 127.0, 0) 

(4, 2, 21, 'U', 2667.0, 1) 

(3, 4, 22, 'L', 2689.0, 1) 

(1, 5, 23, 'L', 2712.0, 1) 

(1, 7, 24, 'D', 113.0, 0) 

(2, 5, 25, 'L', 138.0, 0) 

(3, 3, 26, 'L', 164.0, 0) 

(5, 2, 27, 'L', 191.0, 0) 

(6, 0, 28, 'L', 219.0, 0) 

(7, 2, 29, 'L', 248.0, 0) 

(7, 4, 30, 'U', 7440.0, 1) 

(7, 6, 31, 'D', 240.0, 0) 

(5, 5, 32, 'L', 272.0, 0) 

(5, 7, 33, 'U', 8976.0, 1) 

(3, 7, 34, 'D', 264.0, 0) 

(1, 6, 35, 'L', 299.0, 0) 

(3, 5, 36, 'L', 335.0, 

In [215]:
def calc_orth_adj_sum(board, r, c):
    total = 0

    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
        nr, nc = r + dr, c + dc

        if 0 <= nr < N and 0 <= nc < N:
            total += board[nr][nc] or 0

    return total

In [216]:
total = 0

for r, row in enumerate(final_board_):
    for c, val in enumerate(row):
        if val is None:
            square_sum = calc_orth_adj_sum(final_board_, r, c)
            print("Square", r, c, square_sum)
            total += square_sum

print("Total", total)

Square 0 0 44.0
Square 0 1 574.0
Square 0 6 1436.0
Square 2 1 2012.0
Square 3 6 1646.0
Square 4 6 1890.0
Square 6 7 9925.0
Square 7 3 8392.0
Square 7 5 7690.0
Total 33609.0
